# 04 — Plots & Visualization
All strategies on the same charts. Baselines + RL agent.

Run `02_Run_Baselines.ipynb` and `03_Train_RL.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

RESULTS_DIR = "../Results"
SAVE_PLOTS = True

COLORS = {
    "QQQ Buy-and-Hold":     "#636EFA",
    "Equal-Weight Monthly": "#EF553B",
    "Inverse Volatility":   "#00CC96",
    "Momentum Top-20%":     "#AB63FA",
    "Supervised + MVO":     "#FFA15A",
    "RL Agent":             "#FF6692",
    "QQQ":                  "#636EFA",
}

def get_color(name):
    for key, color in COLORS.items():
        if key in name:
            return color
    return "#19D3F3" 

## 1. Load Data

In [ ]:
# Baselines (full period)
bl_equity = pd.read_csv(f"{RESULTS_DIR}/equity_curves_full.csv", index_col=0, parse_dates=True)
bl_returns = pd.read_csv(f"{RESULTS_DIR}/daily_returns_full.csv", index_col=0, parse_dates=True)
bl_metrics = pd.read_csv(f"{RESULTS_DIR}/performance_metrics_full.csv", index_col=0)
print(f"Baselines: {list(bl_equity.columns)}")

# RL (single stitched OOS equity)
rl_available = os.path.exists(f"{RESULTS_DIR}/rl_equity_oos.csv")
if rl_available:
    rl_equity = pd.read_csv(f"{RESULTS_DIR}/rl_equity_oos.csv", index_col=0, parse_dates=True)
    rl_returns = pd.read_csv(f"{RESULTS_DIR}/rl_daily_returns_oos.csv", index_col=0, parse_dates=True)
    rl_metrics = pd.read_csv(f"{RESULTS_DIR}/rl_performance_metrics.csv", index_col=0)
    rl_fold_log = pd.read_csv(f"{RESULTS_DIR}/rl_fold_log.csv")
    print(f"RL Agent OOS: {rl_equity.index[0]} → {rl_equity.index[-1]} ({len(rl_equity)} days)")
    display(rl_metrics)
else:
    print("RL not yet trained. Run 03_Train_RL.ipynb first.")

## 2. Single Equity Chart — All Strategies
Baselines run on their full period. RL equity is stitched OOS.
Overlapping date range is the fair comparison zone.

In [ ]:
fig = go.Figure()

# Baselines
for col in bl_equity.columns:
    fig.add_trace(go.Scatter(
        x=bl_equity.index, y=bl_equity[col], name=col, mode="lines",
        line=dict(color=get_color(col), width=2.0 if "QQQ" in col else 1.2), opacity=0.7))

# RL Agent (thick, prominent)
if rl_available:
    fig.add_trace(go.Scatter(
        x=rl_equity.index, y=rl_equity["RL Agent"],
        name="RL Agent (OOS)", mode="lines",
        line=dict(color=COLORS["RL Agent"], width=3.0)))

fig.update_layout(
    title="Equity Curves — All Strategies",
    xaxis_title="Date", yaxis_title="Portfolio Value ($1 start)",
    template="plotly_white", height=600,
    legend=dict(x=0.02, y=0.98), hovermode="x unified")
if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_equity_all.html")
fig.show()

## 3. Drawdown — RL vs QQQ

In [ ]:
if rl_available:
    fig = go.Figure()
    for name, eq in [("QQQ", rl_equity["QQQ"]), ("RL Agent", rl_equity["RL Agent"])]:
        dd = (eq - eq.cummax()) / eq.cummax() * 100
        fig.add_trace(go.Scatter(
            x=dd.index, y=dd, name=name, mode="lines",
            line=dict(color=get_color(name), width=2.5 if "RL" in name else 1.5),
            fill="tozeroy" if "QQQ" in name else None,
            opacity=0.3 if "QQQ" in name else 1.0))

    fig.update_layout(title="Drawdown — RL Agent vs QQQ (OOS Period)",
        xaxis_title="Date", yaxis_title="Drawdown (%)",
        template="plotly_white", height=400, hovermode="x unified")
    if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_drawdown_rl.html")
    fig.show()

## 4. Rolling Sharpe — RL vs QQQ

In [ ]:
if rl_available:
    fig = go.Figure()
    window = 63  # ~3 months
    for name, ret in [("QQQ", rl_returns["QQQ"]), ("RL Agent", rl_returns["RL Agent"])]:
        rm = ret.rolling(window).mean()
        rs = ret.rolling(window).std()
        rolling_sharpe = (rm / rs * np.sqrt(252)).dropna()
        fig.add_trace(go.Scatter(
            x=rolling_sharpe.index, y=rolling_sharpe, name=name, mode="lines",
            line=dict(color=get_color(name), width=2.0)))

    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.update_layout(title=f"Rolling {window}-Day Sharpe — RL vs QQQ",
        xaxis_title="Date", yaxis_title="Sharpe",
        template="plotly_white", height=400, hovermode="x unified")
    if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_rolling_sharpe_rl.html")
    fig.show()

## 5. Per-Fold IR2 + Retrain Markers

In [ ]:
if rl_available:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=rl_fold_log["fold_id"], y=rl_fold_log["test_ir2"],
        marker_color=[COLORS["RL Agent"] if r else "#CCCCCC" for r in rl_fold_log["retrained"]],
        name="Test IR2"))
    fig.add_trace(go.Scatter(
        x=rl_fold_log["fold_id"], y=rl_fold_log["val_ir2"],
        mode="lines+markers", name="Val IR2",
        line=dict(color="#636EFA", width=1.5), marker=dict(size=4)))

    # Mark retrains
    retrained = rl_fold_log[rl_fold_log["retrained"]]
    fig.add_trace(go.Scatter(
        x=retrained["fold_id"], y=retrained["test_ir2"],
        mode="markers", name="Retrained",
        marker=dict(size=10, color="red", symbol="triangle-up")))

    fig.update_layout(title="Per-Fold IR2 (colored = retrained, gray = carried)",
        xaxis_title="Fold", yaxis_title="IR2",
        template="plotly_white", height=450)
    if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_fold_ir2.html")
    fig.show()

## 6. Metrics Comparison Bar Chart

In [ ]:
if rl_available:
    combined = pd.concat([bl_metrics, rl_metrics.loc[["RL Agent"]]]).sort_values("IR2", ascending=True)

    fig = make_subplots(rows=1, cols=3, subplot_titles=["IR2", "IR1 (ARC/ASD)", "ARC (%)"], horizontal_spacing=0.12)
    for i, metric in enumerate(["IR2", "IR1", "ARC (%)"]):
        fig.add_trace(go.Bar(
            y=combined.index, x=combined[metric], orientation="h",
            marker_color=[get_color(n) for n in combined.index],
            showlegend=False), row=1, col=i+1)

    fig.update_layout(title="Performance Metrics — All Strategies",
        template="plotly_white", height=400, width=1200)
    if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_metrics_bars.html")
    fig.show()

## 7. LaTeX Tables

In [ ]:
cols = ["ARC (%)", "ASD (%)", "Max Drawdown (%)", "MLD (years)", "IR1", "IR2"]

print("--- Baselines (Full Period) ---")
bl_cols = [c for c in cols if c in bl_metrics.columns]
print(bl_metrics[bl_cols].to_latex(float_format="%.3f",
    caption="Baseline Performance (Full Period, 5 bps TC)", label="tab:baselines"))

if rl_available:
    print("\n--- RL Agent (OOS) ---")
    rl_cols = [c for c in cols if c in rl_metrics.columns]
    print(rl_metrics[rl_cols].to_latex(float_format="%.3f",
        caption="RL Agent OOS Performance (Walk-Forward, 5 bps TC)", label="tab:rl"))

    print("\n--- Combined ---")
    combined = pd.concat([bl_metrics, rl_metrics.loc[["RL Agent"]]]).sort_values("IR2", ascending=False)
    c_cols = [c for c in cols if c in combined.columns]
    print(combined[c_cols].to_latex(float_format="%.3f",
        caption="All Strategies Comparison", label="tab:all"))